# Neighborhood Computations

In [1]:
import numpy as np
import igl
import pyvista as pv
pv.set_jupyter_backend('trame')


In [2]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

In [3]:
bunny_v, bunny_f = igl.read_triangle_mesh("data/bunny.off")
cube_v, cube_f = igl.read_triangle_mesh("data/cube.obj")
sphere_v, sphere_f = igl.read_triangle_mesh("data/sphere.obj")

In [ ]:
# meshplot.plot(cube_v, cube_f, shading={"wireframe": True})
to_pyvista_mesh(bunny_v, bunny_f).plot(show_edges=True)

In [ ]:
to_pyvista_mesh(cube_v, cube_f).plot(show_edges=True)

In [ ]:
to_pyvista_mesh(sphere_v, sphere_f).plot(show_edges=True)

## Vertex-to-Face Relations

In [12]:
VF, VI = igl.vertex_triangle_adjacency(cube_f, cube_v.shape[0])
print(VF)


[ 0  6  7 10 11  0  1  7  8  0  1  2 11  1  2  3  8  9  2  3  4 10 11  3
  4  5  9  4  5  6 10  5  6  7  8  9]


## Vertex-to-Vertex Relations

In [13]:
VV = igl.adjacency_list(cube_f)
print(cube_f)
print(VV)
print(VF)

[[0 1 2]
 [2 1 3]
 [2 3 4]
 [4 3 5]
 [4 5 6]
 [6 5 7]
 [6 7 0]
 [0 7 1]
 [1 7 3]
 [3 7 5]
 [6 0 4]
 [4 0 2]]
[[1, 2, 4, 6, 7], [0, 2, 3, 7], [0, 1, 3, 4], [1, 2, 4, 5, 7], [0, 2, 3, 5, 6], [3, 4, 6, 7], [0, 4, 5, 7], [0, 1, 3, 5, 6]]
[ 0  6  7 10 11  0  1  7  8  0  1  2 11  1  2  3  8  9  2  3  4 10 11  3
  4  5  9  4  5  6 10  5  6  7  8  9]


## Shading

In [14]:
def plot_mesh_with_normals(V, F, N, factor=0.1):
    arrows = pv.vector_poly_data(V, N)
    arrows = arrows.glyph(orient='vectors',scale='mag',factor=factor)

    p = pv.Plotter()
    p.add_mesh(to_pyvista_mesh(V, F), show_edges=True)
    p.add_mesh(arrows)
    p.show()

Pyvista requires per vertex normals, so we need to "explode" the mesh

In [15]:
def explode_mesh(V, F, N):
    
    V = np.asarray(V)
    F = np.asarray(F, dtype=int)
    N = np.asarray(N)

    if N.shape[0] == F.shape[0]:
        normal_mode = "face"
    elif N.shape[0] == V.shape[0]:
        normal_mode = "vertex"

    V_new = []
    F_new = []
    N_new = []

    for fi, face in enumerate(F):  # face = [i0, i1, i2]
        base = len(V_new)
        for vi in face:
            V_new.append(V[vi])
            N_new.append(N[fi] if normal_mode == "face" else N[vi])
        F_new.append([base, base + 1, base + 2])

    return np.asarray(V_new), np.asarray(F_new, dtype=int), np.asarray(N_new)

### Flat Shading

In [ ]:
FN = igl.per_face_normals(cube_v, cube_f)
V_new, F_new, N_new = explode_mesh(cube_v, cube_f, FN)
plot_mesh_with_normals(V_new, F_new, N_new, factor=0.5)

### Per-vertex Shading

In [ ]:
VN = igl.per_vertex_normals(cube_v, cube_f)
V_new, F_new, N_new = explode_mesh(cube_v, cube_f, VN)
plot_mesh_with_normals(V_new, F_new, N_new, factor=0.5)

## Connected Components

In [ ]:
car_v, car_f = igl.read_triangle_mesh("data/car.off")
num, c = igl.facet_components(car_f)
to_pyvista_mesh(car_v, car_f).plot(scalars=c, show_edges=False)

## A simple subdivision scheme

In [ ]:
def mid_point_split(V, F):
    V = np.asarray(V, dtype=float)
    F = np.asarray(F, dtype=int)

    nV = V.shape[0]
    nF = F.shape[0]

    # Compute midpoints/centroids per face
    M = []
    for (i, j, k) in F:
        m = (V[i] + V[j] + V[k]) / 3.0
        M.append(m)
    M = np.asarray(M, dtype=float)

    # Append new points to the vertex array
    Vtmp = np.vstack([V, M])

    # Create 3 triangles per original triangle
    F_star = []
    for fi, (i, j, k) in enumerate(F):
        m_idx = nV + fi
        F_star.append([i, j, m_idx])
        F_star.append([j, k, m_idx])
        F_star.append([k, i, m_idx])

    F_star = np.asarray(F_star, dtype=int)
    return Vtmp, F_star, M


def relocate_vertices(V, F):
    V = np.asarray(V, dtype=float)
    F = np.asarray(F, dtype=int)

    VV = igl.adjacency_list(F)  

    P = V.copy()
    for i, nbrs in enumerate(VV):
        nbrs = np.asarray(nbrs, dtype=int)
        n = len(nbrs)
        if n == 0:
            continue

        alpha = (4.0 - 2.0 * np.cos(2.0 * np.pi / n)) / 9.0
        P[i] = (1.0 - alpha) * V[i] + (alpha / n) * V[nbrs].sum(axis=0)

    return P


def flip_original_edges(F_star, nV):
    F_star = np.asarray(F_star, dtype=int)

    edge_to_opposites = {}

    for (a, b, c) in F_star:
        for u, v, w in [(a, b, c), (b, c, a), (c, a, b)]:
            if u < nV and v < nV:  # u and v are original vertices
                key = (min(u,v), max(u,v))
                edge_to_opposites.setdefault(key, []).append(w)  # w is midpoint index

    tris = []
    for (u, v), opp in edge_to_opposites.items():
        mL, mR = opp[0], opp[1]
        tris.append([u, mL, mR])  # flip edges
        tris.append([v, mR, mL])

    return np.asarray(tris, dtype=int)


def subdivide_once(V, F):
    V = np.asarray(V, dtype=float)
    F = np.asarray(F, dtype=int)
    nV = V.shape[0]

    # Create mid-point split
    Vtmp, F_star, M = mid_point_split(V, F)

    # Relocate original vertices
    P = relocate_vertices(V, F)
    Vtmp[:nV] = P  # overwrite old positions with relocated ones
    Vprime = Vtmp  

    # Flip original edges to the new diagonals
    Fprime = flip_original_edges(F_star, nV)

    return Vprime, Fprime

Vtmp, F_star, M = mid_point_split(cube_v, cube_f)
to_pyvista_mesh(Vtmp, F_star).plot(show_edges=True)

V1, F1 = subdivide_once(cube_v, cube_f)
to_pyvista_mesh(V1, F1).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:64092/index.html?ui=P_0x1a651925a90_4&reconnect=auto" class="pyvis…

Widget(value='<iframe src="http://localhost:64092/index.html?ui=P_0x1a6540f02d0_5&reconnect=auto" class="pyvis…